In [ ]:
"""
Create a reproducible synthetic SQLite database for river water-quality monitoring.

The records are fictional and are generated with Python's standard library.
"""

from datetime import datetime, timedelta
from pathlib import Path
import math
import random
import sqlite3


# A fixed seed makes the generated records reproducible.
SEED = 20261103
random.seed(SEED)

# Jupyter and Colab usually do not define __file__, so use the current
# working folder in those environments.
try:
    SCRIPT_FOLDER = Path(__file__).resolve().parent
except NameError:
    SCRIPT_FOLDER = Path.cwd()

DATABASE_PATH = SCRIPT_FOLDER / "river_water_quality.sqlite"

SITE_COUNT = 12
SENSORS_PER_SITE = 3
SAMPLE_COUNT = 1_600
MISSING_MEASUREMENT_COUNT = 90
DUPLICATE_SUBMISSION_COUNT = 32

SAMPLE_START = datetime(2025, 1, 1)
SAMPLE_DAYS = 365

SITE_TYPES = ("upstream", "urban", "agricultural", "downstream")

# These are illustrative ranges for the synthetic data, not official safety limits.
PARAMETERS = (
    ("dissolved_oxygen", "mg/L", 0.0, 16.0),
    ("turbidity", "NTU", 0.0, 200.0),
    ("nitrate", "mg/L", 0.0, 20.0),
    ("conductivity", "uS/cm", 0.0, 2500.0),
)


def clamp(value, minimum, maximum):
    """Keep a generated measurement inside its chosen range."""
    return max(minimum, min(value, maximum))


def create_schema(connection):
    """Create the database tables, keys, and useful indexes."""

    # Child tables are dropped before their parent tables so the script can
    # recreate the database cleanly when it is run again.
    connection.executescript(
        """
        DROP TABLE IF EXISTS sample_ingestions;
        DROP TABLE IF EXISTS sample_measurements;
        DROP TABLE IF EXISTS water_samples;
        DROP TABLE IF EXISTS water_parameters;
        DROP TABLE IF EXISTS sensors;
        DROP TABLE IF EXISTS river_sites;

        CREATE TABLE river_sites (
            site_id INTEGER PRIMARY KEY,
            site_name TEXT NOT NULL UNIQUE,
            river_section TEXT NOT NULL,
            site_type TEXT NOT NULL
                CHECK (site_type IN (
                    'upstream', 'urban', 'agricultural', 'downstream'
                )),
            priority_level INTEGER NOT NULL
                CHECK (priority_level BETWEEN 1 AND 4)
        );

        CREATE TABLE sensors (
            sensor_id INTEGER PRIMARY KEY,
            site_id INTEGER NOT NULL
                REFERENCES river_sites(site_id),
            sensor_code TEXT NOT NULL UNIQUE,
            sensor_type TEXT NOT NULL,
            installed_at TEXT NOT NULL,
            sensor_status TEXT NOT NULL
                CHECK (sensor_status IN (
                    'active', 'maintenance', 'retired'
                )),
            UNIQUE (sensor_id, site_id)
        );

        CREATE TABLE water_parameters (
            parameter_id INTEGER PRIMARY KEY,
            parameter_code TEXT NOT NULL UNIQUE,
            unit TEXT NOT NULL,
            minimum_value REAL NOT NULL,
            maximum_value REAL NOT NULL,
            CHECK (minimum_value < maximum_value)
        );

        CREATE TABLE water_samples (
            sample_id INTEGER PRIMARY KEY,
            site_id INTEGER NOT NULL
                REFERENCES river_sites(site_id),
            sensor_id INTEGER NOT NULL,
            sampled_at TEXT NOT NULL,
            water_temp_c REAL NOT NULL
                CHECK (water_temp_c BETWEEN -5 AND 40),
            flow_rate_m3_s REAL NOT NULL
                CHECK (flow_rate_m3_s BETWEEN 0 AND 100),
            quality_grade INTEGER NOT NULL
                CHECK (quality_grade BETWEEN 1 AND 5),
            sample_status TEXT NOT NULL
                CHECK (sample_status IN ('routine', 'review')),
            FOREIGN KEY (sensor_id, site_id)
                REFERENCES sensors(sensor_id, site_id)
        );

        CREATE TABLE sample_measurements (
            sample_id INTEGER NOT NULL
                REFERENCES water_samples(sample_id),
            parameter_id INTEGER NOT NULL
                REFERENCES water_parameters(parameter_id),
            result_value REAL,
            result_status TEXT NOT NULL
                CHECK (result_status IN ('measured', 'missing')),
            PRIMARY KEY (sample_id, parameter_id),
            CHECK (
                (result_value IS NULL AND result_status = 'missing')
                OR
                (result_value IS NOT NULL AND result_status = 'measured')
            )
        );

        CREATE TABLE sample_ingestions (
            submission_id INTEGER PRIMARY KEY,
            source_reference TEXT NOT NULL,
            sample_id INTEGER NOT NULL
                REFERENCES water_samples(sample_id),
            submitted_at TEXT NOT NULL,
            duplicate_of_submission_id INTEGER
                REFERENCES sample_ingestions(submission_id),
            ingestion_status TEXT NOT NULL
                CHECK (ingestion_status IN ('original', 'duplicate')),
            CHECK (
                (ingestion_status = 'original'
                    AND duplicate_of_submission_id IS NULL)
                OR
                (ingestion_status = 'duplicate'
                    AND duplicate_of_submission_id IS NOT NULL)
            )
        );

        CREATE INDEX idx_samples_site_time
            ON water_samples(site_id, sampled_at);

        CREATE INDEX idx_ingestions_sample
            ON sample_ingestions(sample_id);
        """
    )


def create_sites(connection):
    """Add fictional monitoring sites along the river."""
    site_rows = []

    for site_id in range(1, SITE_COUNT + 1):
        site_type = SITE_TYPES[(site_id - 1) % len(SITE_TYPES)]

        if site_id <= 4:
            river_section = "upper reach"
        elif site_id <= 8:
            river_section = "middle reach"
        else:
            river_section = "lower reach"

        priority = {
            "upstream": 1,
            "urban": 3,
            "agricultural": 2,
            "downstream": 4,
        }[site_type]

        site_rows.append(
            (
                site_id,
                f"River Monitoring Site {site_id:02d}",
                river_section,
                site_type,
                priority,
            )
        )

    connection.executemany(
        """
        INSERT INTO river_sites
            (site_id, site_name, river_section, site_type, priority_level)
        VALUES (?, ?, ?, ?, ?)
        """,
        site_rows,
    )


def create_sensors(connection):
    """Place three sensors at each fictional monitoring site."""
    sensor_rows = []
    sensors_by_site = {}
    sensor_id = 1

    for site_id in range(1, SITE_COUNT + 1):
        sensors_by_site[site_id] = []

        for sensor_number in range(1, SENSORS_PER_SITE + 1):
            install_date = datetime(2024, 1, 1) + timedelta(
                days=random.randint(0, 330)
            )

            sensor_rows.append(
                (
                    sensor_id,
                    site_id,
                    f"SEN-{site_id:02d}-{sensor_number:02d}",
                    "multi-parameter probe",
                    install_date.strftime("%Y-%m-%d"),
                    "active",
                )
            )
            sensors_by_site[site_id].append(sensor_id)
            sensor_id += 1

    connection.executemany(
        """
        INSERT INTO sensors
            (sensor_id, site_id, sensor_code, sensor_type,
             installed_at, sensor_status)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        sensor_rows,
    )

    return sensors_by_site


def create_parameters(connection):
    """Store parameter names and units once, rather than repeating them."""
    parameter_rows = []

    for parameter_id, (code, unit, minimum, maximum) in enumerate(
        PARAMETERS, start=1
    ):
        parameter_rows.append(
            (parameter_id, code, unit, minimum, maximum)
        )

    connection.executemany(
        """
        INSERT INTO water_parameters
            (parameter_id, parameter_code, unit, minimum_value, maximum_value)
        VALUES (?, ?, ?, ?, ?)
        """,
        parameter_rows,
    )

    return {
        code: parameter_id
        for parameter_id, (code, _, _, _) in enumerate(PARAMETERS, start=1)
    }


def generate_measurements(site_type, water_temp_c):
    """Generate readings with modest differences between site settings."""
    site_effect = {
        "upstream": 0.0,
        "urban": 1.5,
        "agricultural": 1.0,
        "downstream": 0.8,
    }[site_type]

    dissolved_oxygen = clamp(
        random.gauss(
            9.0 - (water_temp_c - 12) * 0.15 - site_effect,
            1.8,
        ),
        0.2,
        15.5,
    )

    turbidity_median = {
        "upstream": 5.0,
        "urban": 13.0,
        "agricultural": 16.0,
        "downstream": 9.0,
    }[site_type]
    turbidity = clamp(
        random.lognormvariate(math.log(turbidity_median), 0.55),
        0.2,
        190.0,
    )

    nitrate_mean = {
        "upstream": 1.2,
        "urban": 3.0,
        "agricultural": 5.5,
        "downstream": 2.5,
    }[site_type]
    nitrate = clamp(
        random.gauss(nitrate_mean, 1.5),
        0.0,
        19.5,
    )

    conductivity_mean = {
        "upstream": 250.0,
        "urban": 650.0,
        "agricultural": 800.0,
        "downstream": 500.0,
    }[site_type]
    conductivity = clamp(
        random.gauss(conductivity_mean, 180.0),
        20.0,
        2450.0,
    )

    return {
        "dissolved_oxygen": round(dissolved_oxygen, 2),
        "turbidity": round(turbidity, 2),
        "nitrate": round(nitrate, 2),
        "conductivity": round(conductivity, 2),
    }


def grade_sample(measurements):
    """Assign an ordinal grade using the generated sample readings."""
    score = 3

    if measurements["dissolved_oxygen"] < 5:
        score -= 2
    elif measurements["dissolved_oxygen"] >= 9:
        score += 1

    if measurements["turbidity"] > 60:
        score -= 2
    elif measurements["turbidity"] < 10:
        score += 1

    if measurements["nitrate"] > 8:
        score -= 1
    elif measurements["nitrate"] < 2:
        score += 1

    return max(1, min(score, 5))


def create_samples_and_measurements(
    connection,
    sensors_by_site,
    parameter_ids,
):
    """Create 1,600 samples and their linked measurement records."""
    site_types = dict(
        connection.execute(
            "SELECT site_id, site_type FROM river_sites"
        ).fetchall()
    )

    sample_rows = []
    raw_measurements_by_sample = {}

    for sample_id in range(1, SAMPLE_COUNT + 1):
        # Cycle through sites to give each one a useful number of observations.
        site_id = ((sample_id - 1) % SITE_COUNT) + 1
        sensor_id = random.choice(sensors_by_site[site_id])
        site_type = site_types[site_id]

        sampled_at = SAMPLE_START + timedelta(
            days=random.randrange(SAMPLE_DAYS),
            hours=random.randrange(24),
            minutes=random.randrange(60),
        )

        # Temperature changes with the season, with a small amount of variation.
        day_of_year = sampled_at.timetuple().tm_yday
        seasonal_temperature = 12 + 8 * math.sin(
            2 * math.pi * (day_of_year - 100) / 365
        )
        water_temp_c = round(
            clamp(
                seasonal_temperature + random.uniform(-2.5, 2.5),
                -2,
                30,
            ),
            2,
        )

        flow_multiplier = {
            "upstream": 0.7,
            "urban": 1.0,
            "agricultural": 1.1,
            "downstream": 1.8,
        }[site_type]
        flow_rate = round(
            random.uniform(0.5, 12.0) * flow_multiplier,
            2,
        )

        measurements = generate_measurements(site_type, water_temp_c)
        quality_grade = grade_sample(measurements)
        sample_status = "review" if quality_grade <= 2 else "routine"

        sample_rows.append(
            (
                sample_id,
                site_id,
                sensor_id,
                sampled_at.strftime("%Y-%m-%d %H:%M:%S"),
                water_temp_c,
                flow_rate,
                quality_grade,
                sample_status,
            )
        )
        raw_measurements_by_sample[sample_id] = measurements

    connection.executemany(
        """
        INSERT INTO water_samples
            (sample_id, site_id, sensor_id, sampled_at, water_temp_c,
             flow_rate_m3_s, quality_grade, sample_status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        sample_rows,
    )

    # Each sample can have results for several parameters. The composite key
    # prevents the same parameter being entered twice for one sample.
    all_keys = [
        (sample_id, parameter_id)
        for sample_id in range(1, SAMPLE_COUNT + 1)
        for parameter_id in range(1, len(PARAMETERS) + 1)
    ]
    missing_keys = set(
        random.sample(all_keys, MISSING_MEASUREMENT_COUNT)
    )

    measurement_rows = []

    for sample_id, parameter_id in all_keys:
        parameter_code = next(
            code
            for code, mapped_id in parameter_ids.items()
            if mapped_id == parameter_id
        )
        value = raw_measurements_by_sample[sample_id][parameter_code]

        if (sample_id, parameter_id) in missing_keys:
            # A missing value represents a sensor or test result that was unavailable.
            measurement_rows.append(
                (sample_id, parameter_id, None, "missing")
            )
        else:
            measurement_rows.append(
                (sample_id, parameter_id, value, "measured")
            )

    connection.executemany(
        """
        INSERT INTO sample_measurements
            (sample_id, parameter_id, result_value, result_status)
        VALUES (?, ?, ?, ?)
        """,
        measurement_rows,
    )


def create_ingestions(connection):
    """Record original submissions and 32 deliberate repeat submissions."""
    original_submission_ids = {}
    source_references = {}

    samples = connection.execute(
        """
        SELECT sample_id, sampled_at
        FROM water_samples
        ORDER BY sample_id
        """
    ).fetchall()

    for sample_id, sampled_at in samples:
        source_reference = f"SRC-SAMPLE-{sample_id:05d}"
        submitted_at = datetime.strptime(
            sampled_at, "%Y-%m-%d %H:%M:%S"
        ) + timedelta(minutes=random.randint(5, 90))

        cursor = connection.execute(
            """
            INSERT INTO sample_ingestions
                (source_reference, sample_id, submitted_at,
                 duplicate_of_submission_id, ingestion_status)
            VALUES (?, ?, ?, NULL, 'original')
            """,
            (
                source_reference,
                sample_id,
                submitted_at.strftime("%Y-%m-%d %H:%M:%S"),
            ),
        )

        original_submission_ids[sample_id] = cursor.lastrowid
        source_references[sample_id] = source_reference

    # The repeat keeps the same source and sample identifiers, but receives
    # its own submission ID and a link back to the original submission.
    repeated_sample_ids = random.sample(
        range(1, SAMPLE_COUNT + 1),
        DUPLICATE_SUBMISSION_COUNT,
    )

    for sample_id in repeated_sample_ids:
        original_id = original_submission_ids[sample_id]
        original_time = connection.execute(
            """
            SELECT submitted_at
            FROM sample_ingestions
            WHERE submission_id = ?
            """,
            (original_id,),
        ).fetchone()[0]

        repeated_time = datetime.strptime(
            original_time, "%Y-%m-%d %H:%M:%S"
        ) + timedelta(minutes=random.randint(1, 30))

        connection.execute(
            """
            INSERT INTO sample_ingestions
                (source_reference, sample_id, submitted_at,
                 duplicate_of_submission_id, ingestion_status)
            VALUES (?, ?, ?, ?, 'duplicate')
            """,
            (
                source_references[sample_id],
                sample_id,
                repeated_time.strftime("%Y-%m-%d %H:%M:%S"),
                original_id,
            ),
        )


def print_schema_and_counts(connection):
    """Display every table's row count, columns, keys, and nullability."""
    table_names = (
        "river_sites",
        "sensors",
        "water_parameters",
        "water_samples",
        "sample_measurements",
        "sample_ingestions",
    )

    print("\nTABLE COUNTS AND SCHEMA")

    for table_name in table_names:
        row_count = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]
        columns = connection.execute(
            f'PRAGMA table_info("{table_name}")'
        ).fetchall()
        foreign_keys = connection.execute(
            f'PRAGMA foreign_key_list("{table_name}")'
        ).fetchall()

        print(f"\n{table_name}: {row_count:,} rows")

        for column in columns:
            name = column[1]
            data_type = column[2]
            is_required = bool(column[3])
            primary_key_order = column[5]

            labels = []
            if primary_key_order:
                labels.append("PK")
            if is_required or primary_key_order:
                labels.append("NOT NULL")
            else:
                labels.append("NULL allowed")

            print(f"  {name} {data_type} ({', '.join(labels)})")

        for foreign_key in foreign_keys:
            print(
                f"  FK: {foreign_key[3]} -> "
                f"{foreign_key[2]}.{foreign_key[4]}"
            )


def validate_database(connection):
    """Report the intentional data-quality cases and check database integrity."""
    sample_column_count = len(
        connection.execute(
            "PRAGMA table_info('water_samples')"
        ).fetchall()
    )
    sample_count = connection.execute(
        "SELECT COUNT(*) FROM water_samples"
    ).fetchone()[0]

    missing_count, measurement_count = connection.execute(
        """
        SELECT
            SUM(CASE WHEN result_value IS NULL THEN 1 ELSE 0 END),
            COUNT(*)
        FROM sample_measurements
        """
    ).fetchone()

    duplicate_groups = connection.execute(
        """
        SELECT COUNT(*)
        FROM (
            SELECT source_reference, sample_id
            FROM sample_ingestions
            GROUP BY source_reference, sample_id
            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0]

    duplicate_extra_rows = connection.execute(
        """
        SELECT COALESCE(SUM(group_size - 1), 0)
        FROM (
            SELECT COUNT(*) AS group_size
            FROM sample_ingestions
            GROUP BY source_reference, sample_id
            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0]

    invalid_measurements = connection.execute(
        """
        SELECT COUNT(*)
        FROM sample_measurements AS m
        JOIN water_parameters AS p
          ON p.parameter_id = m.parameter_id
        WHERE m.result_value IS NOT NULL
          AND (
              m.result_value < p.minimum_value
              OR m.result_value > p.maximum_value
          )
        """
    ).fetchone()[0]

    invalid_dates = connection.execute(
        """
        SELECT COUNT(*)
        FROM water_samples
        WHERE date(sampled_at) IS NULL
        """
    ).fetchone()[0]

    mismatched_sensor_sites = connection.execute(
        """
        SELECT COUNT(*)
        FROM water_samples AS ws
        JOIN sensors AS s ON s.sensor_id = ws.sensor_id
        WHERE ws.site_id <> s.site_id
        """
    ).fetchone()[0]

    integrity_result = connection.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]
    foreign_key_errors = connection.execute(
        "PRAGMA foreign_key_check"
    ).fetchall()

    print("\nDATA-QUALITY SUMMARY")
    print(
        f"Missing measurements: {missing_count:,}/{measurement_count:,} "
        f"({missing_count / measurement_count * 100:.2f}%)"
    )
    print(
        f"Duplicate submissions: {duplicate_extra_rows:,} extra rows in "
        f"{duplicate_groups:,} repeated groups "
        f"({duplicate_extra_rows / SAMPLE_COUNT * 100:.2f}% "
        "of original submissions)"
    )

    print("\nVALIDATION")
    meets_minimum = sample_count >= 1_000 and sample_column_count >= 7
    print(f"water_samples meets the 1,000-row and 7-column minimum: {meets_minimum}")
    print(f"Invalid measurement ranges: {invalid_measurements}")
    print(f"Invalid sample dates: {invalid_dates}")
    print(f"Sensor/site mismatches: {mismatched_sensor_sites}")
    print(f"SQLite integrity check: {integrity_result}")
    print(f"Foreign-key violations: {len(foreign_key_errors)}")

    problems = []

    if not meets_minimum:
        problems.append("water_samples does not meet the minimum size requirement.")
    if missing_count != MISSING_MEASUREMENT_COUNT:
        problems.append("The missing-measurement count is unexpected.")
    if duplicate_extra_rows != DUPLICATE_SUBMISSION_COUNT:
        problems.append("The duplicate-submission count is unexpected.")
    if invalid_measurements:
        problems.append("At least one measurement is outside its defined range.")
    if invalid_dates:
        problems.append("At least one sample has an invalid date.")
    if mismatched_sensor_sites:
        problems.append("At least one sample uses a sensor from another site.")
    if integrity_result != "ok":
        problems.append("SQLite reported an integrity issue.")
    if foreign_key_errors:
        problems.append("SQLite reported a foreign-key violation.")

    if problems:
        raise RuntimeError("\n".join(problems))


def main():
    connection = sqlite3.connect(DATABASE_PATH)

    # SQLite requires this setting on each connection before data is inserted.
    connection.execute("PRAGMA foreign_keys = ON")

    try:
        create_schema(connection)
        create_sites(connection)
        sensors_by_site = create_sensors(connection)
        parameter_ids = create_parameters(connection)

        create_samples_and_measurements(
            connection,
            sensors_by_site,
            parameter_ids,
        )
        create_ingestions(connection)
        connection.commit()

        print(f"Database created: {DATABASE_PATH}")
        print(f"Random seed: {SEED}")
        print_schema_and_counts(connection)
        validate_database(connection)
    finally:
        connection.close()


if __name__ == "__main__":
    main()

Database created: /content/river_water_quality.sqlite
Random seed: 20261103

TABLE COUNTS AND SCHEMA

river_sites: 12 rows
  site_id INTEGER (PK, NOT NULL)
  site_name TEXT (NOT NULL)
  river_section TEXT (NOT NULL)
  site_type TEXT (NOT NULL)
  priority_level INTEGER (NOT NULL)

sensors: 36 rows
  sensor_id INTEGER (PK, NOT NULL)
  site_id INTEGER (NOT NULL)
  sensor_code TEXT (NOT NULL)
  sensor_type TEXT (NOT NULL)
  installed_at TEXT (NOT NULL)
  sensor_status TEXT (NOT NULL)
  FK: site_id -> river_sites.site_id

water_parameters: 4 rows
  parameter_id INTEGER (PK, NOT NULL)
  parameter_code TEXT (NOT NULL)
  unit TEXT (NOT NULL)
  minimum_value REAL (NOT NULL)
  maximum_value REAL (NOT NULL)

water_samples: 1,600 rows
  sample_id INTEGER (PK, NOT NULL)
  site_id INTEGER (NOT NULL)
  sensor_id INTEGER (NOT NULL)
  sampled_at TEXT (NOT NULL)
  water_temp_c REAL (NOT NULL)
  flow_rate_m3_s REAL (NOT NULL)
  quality_grade INTEGER (NOT NULL)
  sample_status TEXT (NOT NULL)
  FK: sensor